In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, when, substring, length, lit,
    to_date, to_timestamp, datediff, dayofweek
)
from murph import mosaic

od_mile_path = "dbfs:/Users/939510@corpaa.aa.com/od_distance_market.csv"
df_spark_od_mile = (spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(od_mile_path)
            .filter(
                ~F.col("Market").contains("OFF") &
                (F.col("Market") != "MC-MXoo-BUS")
            ))

df_spark_od_mile.createOrReplaceTempView("od_mile_market")


# Load and broadcast small lookup tables
from pyspark.sql.functions import broadcast
from pyspark.storagelevel import StorageLevel

od_mile_df = spark.table("od_mile_market").persist(StorageLevel.MEMORY_AND_DISK)
_ = od_mile_df.count()
broadcast(od_mile_df).createOrReplaceTempView("od_mile_market")

# Leg-level Flight Frequency
# Load OD frequency table
spark.table("rm_workspace.OD_Market_Frequency").createOrReplaceTempView("od_freq")
od_freq_df = spark.table("od_freq").persist(StorageLevel.MEMORY_AND_DISK)
_ = od_freq_df.count()
broadcast(od_freq_df).createOrReplaceTempView("leg_freq")


# OD Market Region
# Load OD market region table
spark.table("rm_workspace.OD_Market_Region").createOrReplaceTempView("od_market_region")
od_market_region_df = spark.table("od_market_region").persist(StorageLevel.MEMORY_AND_DISK)
_ = od_market_region_df.count()
broadcast(od_market_region_df).createOrReplaceTempView("od_market_region")


user = "939510"
mos = mosaic(user = user, group_name = "group106", tmode = "TERA")


def razor(df):
  for col in df.dtypes:
    if col[1]=='string':
      df = df.withColumn(col[0],trim(col[0]))
  return df

In [0]:
df = spark.sql("SHOW TABLES IN prod_mod_pkg_pii")
all_table_names = [r.tableName for r in df.select("tableName").collect()]
table_names =[x for x in all_table_names if 'anc_offer' in x]
table_names

'''
03-11-2026
- Ideally, we would want to consider seat, IU, Priority, SDFC, but it might be too big. We should consider a specific time range
- Start w/ data building with anc_offer_sale_seat for proof of concept
-- 
'''
from datetime import datetime, timedelta


# Things to update whenever run
# start_date = max(dataPartition) + 1 from rm_workspace.finalTransactionOfferSale_B
# end_date = User defined.. 
test_run = True
start_date ='2025-01-01'
end_date = '2026-06-30'


# max 20250604 with pfactor'%Adjustment4%'

### prod_mod_pkg_pii.anc_offer_pnr_priority
- Contains a set of PRN-level offers (shown to PNRs, Avg, Max, Min, Mode), Min/Max offer dates, and sale date/time, and sale indicator. 

### prod_mod_pkg_pii.anc_offer_sale_priority
- Only contains sale data (the offer that made it to sales)


### prod_mod_pkg_pii.anc_offer_priority


In [0]:
from datetime import date
if test_run == False:
    itinerary_sql = f"""
    SELECT
        a.PNR_LOCTR_ID,
        a.PNR_CREATE_DT,
        a.OD_DEP_AIRPRT_IATA_CD,
        a.OD_ARVL_AIRPRT_IATA_CD,
        a.OD_DEP_DT,
        a.L1_DEP_TM,
        e.CITY_METRO_IATA_CD        AS origin_city,
        f.CITY_METRO_IATA_CD        AS destination_city,
        e.CNTRY_CD                  AS origin_country,
        f.CNTRY_CD                  AS destination_country,
        MAX(CASE
            WHEN e.MIRS_PRIORITY_CD > f.MIRS_PRIORITY_CD
                THEN e.MIRS_PRIORITY_REGION_DESC
            ELSE f.MIRS_PRIORITY_REGION_DESC
        END)                        AS region, 
        AVG(BUSINES_ONLY_PROB_PCT) AS BUSINESS_PROB,
        AVG(BLEISURE_PROB_PCT) AS BLEISURE_PROB,
        AVG(VFR_PROB_PCT) AS VFR_PROB,
        AVG(VACTN_ONLY_PROB_PCT) AS VACATION_PROB,
        AVG(PERSNL_OTHR_PROB_PCT) AS PERSONAL_PROB

    FROM PROD_RM_BUSINES_VW.RBK_OD_INTK a

    LEFT JOIN PAXORDMSTR_PROD_PKG.PNR_MASTER r
        ON a.PNR_LOCTR_ID = r.PNR_LOCTR_ID
        AND a.PNR_CREATE_DT = r.PNR_CREATE_DT

    LEFT JOIN PROD_REFERENCE_DATA_VWS.AIRPORT_STATION_CURRENT e
        ON a.OD_DEP_AIRPRT_IATA_CD = e.AIRPRT_CD

    LEFT JOIN PROD_REFERENCE_DATA_VWS.AIRPORT_STATION_CURRENT f
        ON a.OD_ARVL_AIRPRT_IATA_CD = f.AIRPRT_CD


    WHERE
        a.PNR_CREATE_DT BETWEEN DATE '{start_date}' AND DATE '{end_date}'
        AND a.PNR_LOCTR_ID IS NOT NULL

    GROUP BY
    a.PNR_LOCTR_ID,
    a.PNR_CREATE_DT,
    a.OD_DEP_AIRPRT_IATA_CD,
    a.OD_ARVL_AIRPRT_IATA_CD,
    a.OD_DEP_DT,
    a.L1_DEP_TM,
    e.CITY_METRO_IATA_CD,
    f.CITY_METRO_IATA_CD,
    e.CNTRY_CD,
    f.CNTRY_CD
    """

    import time
    t0 = time.time()
    itinerary = razor(mos.read_sql(itinerary_sql))
    itinerary = itinerary.select([col(x).alias(x.lower()) for x in itinerary.columns])

    itinerary.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("rm_workspace.tmp_rbk_data")
    print(f"✓ done in {(time.time()-t0)/60:.1f} min")

itinerary = (
spark.table("rm_workspace.tmp_rbk_data")
)

itinerary.createOrReplaceTempView("itinerary")
print("\u2713 itinerary ready")

In [0]:
# itinerary is OD-level after GROUP BY — result is small (one row per OD pair).
# Materializing it once here avoids re-computing the same CTE in both enrichment cells (13 and 17).
if test_run:
    spark.sql("""
        WITH market_profile AS (
            SELECT
                od_dep_airprt_iata_cd AS od_origin,
                od_arvl_airprt_iata_cd AS od_destination,
                COUNT(*) AS pnr_od_count,
                MAX(region) AS region,
                MAX(origin_country) AS origin_country,
                MAX(destination_country) AS destination_country,
                AVG(COALESCE(business_prob, 0)) AS avg_business_prob,
                AVG(COALESCE(bleisure_prob, 0)) AS avg_bleisure_prob,
                AVG(COALESCE(vfr_prob, 0)) AS avg_vfr_prob,
                AVG(COALESCE(vacation_prob, 0)) AS avg_vacation_prob,
                AVG(COALESCE(personal_prob, 0)) AS avg_personal_prob
            FROM itinerary
            GROUP BY od_dep_airprt_iata_cd, od_arvl_airprt_iata_cd
        )
        SELECT *,
            CASE
                WHEN (avg_business_prob + avg_bleisure_prob) >= avg_vfr_prob
                 AND (avg_business_prob + avg_bleisure_prob) >= (avg_vacation_prob + avg_personal_prob)
                 THEN 'Business_Market'
                WHEN avg_vfr_prob >= (avg_business_prob + avg_bleisure_prob)
                 AND avg_vfr_prob >= (avg_vacation_prob + avg_personal_prob)
                 THEN 'VFR_Market'
                ELSE 'Leisure_Market'
            END AS market_traveler_segment
        FROM market_profile
    """).write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("rm_workspace.market_segment")
    print("\u2713 market_segment materialized")

spark.table("rm_workspace.market_segment").createOrReplaceTempView("market_segment")

In [0]:
if test_run:
    keyIDAssignment = spark.sql(f"""
    SELECT DISTINCT datePartition, key_id, unique_id, transactionId, 
           os_displayPrice_currency, os_displayPrice, os_displayTotal, source_type 
    FROM prod_mod_gold_pii.offering
    WHERE os_carrier = 'AA'
        AND os_commercialName LIKE '%Priority%'
        AND datePartition BETWEEN {start_date.replace('-','')} AND {end_date.replace('-','')}
    """)
    keyIDAssignment.write.mode("overwrite").saveAsTable("rm_workspace.keyIDAssignment")
    keyIDAssignment = spark.table("rm_workspace.keyIDAssignment")
    keyIDAssignment.createOrReplaceTempView("keyIDAssignment")
else:
    keyIDAssignment = spark.table("rm_workspace.keyIDAssignment")
    keyIDAssignment.createOrReplaceTempView("keyIDAssignment")

print("\u2713 keyIDAssignment ready")

In [0]:
temp_view = spark.sql(f"""
select t.*, 
        regexp_replace(t.pass_id,'^0' ,'') as pass_id_clean,
        CASE WHEN t.seg_1_flight is null THEN 1
            WHEN t.seg_2_flight is null THEN 2
            ELSE 3
        END as numConnections,
        k.key_id,
        k.unique_id
 From prod_mod_pkg_pii.anc_offer_priority t
 left join keyIDAssignment k
on t.transactionId = k.transactionId
and t.datePartition = k.datePartition
and t.os_displayPrice_currency = k.os_displayPrice_currency
and t.os_displayPrice = k.os_displayPrice
WHERE t.datePartition BETWEEN {start_date.replace('-','')} AND {end_date.replace('-','')}
 """)
if test_run:
    temp_view.write.mode("overwrite") \
             .option("overwriteSchema", "true") \
             .saveAsTable("rm_workspace.tmp_anc_offer_priority")
    print("\u2713 tmp_anc_offer_priority materialized")

# Re-point temp view to the materialized table so cells 7 and 9 read from Delta,
# not by re-scanning prod_mod_pkg_pii.anc_offer_priority each time.
spark.table("rm_workspace.tmp_anc_offer_priority").createOrReplaceTempView("temp_view")

In [0]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from datetime import date

if test_run:
    priority_adj = spark.sql(f"""
    WITH priority_factors AS (
    SELECT 
        transactionId,
        key_id,
        unique_id,
        datePartition,
        CAST(REGEXP_REPLACE(
        MAX(CASE WHEN p_os_annotations_name LIKE '%Base_t%' THEN p_os_annotations_values END),
        '[^0-9.]', '') AS DOUBLE) AS base_price,
        CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Surcharge_t%' THEN p_os_annotations_values END) AS DOUBLE) AS surcharge,
        CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Security%' THEN p_os_annotations_values END) AS DOUBLE) AS security_line,
        CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%DOW%' THEN p_os_annotations_values END) AS DOUBLE) AS dow_time,
        CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Functionality%' THEN p_os_annotations_values END) AS DOUBLE) AS functionality
    FROM prod_mod_gold_pii.annotative_price
    WHERE product_type <> 'seats'
        AND (p_os_annotations_name LIKE '%Priority%' OR p_os_annotations_name LIKE '%priority%')
        AND datePartition BETWEEN {start_date.replace('-','')} AND {end_date.replace('-','')}
    GROUP BY transactionId, datePartition, key_id, unique_id
    )
    SELECT 
    t.datePartition, 
    t.transactionId, 
    t.os_displayPrice, 
    t.os_displayPrice_currency, 
    t.numConnections,
    t.key_id,
    t.unique_id,
    pf.base_price,
    pf.surcharge,
    pf.security_line,
    pf.dow_time,
    pf.functionality,
    CASE 
        WHEN t.os_displayPrice_currency = 'USD' THEN t.os_displayPrice
        ELSE (pf.base_price * pf.surcharge * pf.security_line * pf.dow_time * pf.functionality) + 2*t.numConnections
    END AS adj_displayPrice
    FROM temp_view t
    LEFT JOIN priority_factors pf
    ON t.transactionId = pf.transactionId
    AND t.datePartition = pf.datePartition
    AND t.key_id = pf.key_id
    AND t.unique_id = pf.unique_id
    WHERE t.datePartition BETWEEN {start_date.replace('-','')} AND {end_date.replace('-','')}
      AND (t.os_displayPrice_currency != 'USD' OR t.os_displayPrice IS NULL)
      -- cell 11 only uses priority_adj for non-USD rows. Pre-filtering here reduces
      -- the temp_view scan from ~2.3B rows to ~1-5% of that, making this cell
      -- and the resulting priority_adj table dramatically smaller.
    """)

    priority_adj.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("rm_workspace.priority_adj")
    priority_adj = spark.table("rm_workspace.priority_adj")
    priority_adj.createOrReplaceTempView("priority_adj")
else:
    priority_adj = spark.table("rm_workspace.priority_adj")
    priority_adj.createOrReplaceTempView("priority_adj")

In [0]:
# %sql
# create or replace temp view pnr_sale_priority as
# -- select distinct PNR, PNR_CREATE_DT, od_origin, od_destination, od_dep_dt, sale_DT, Sale_Price, os_displayPrice_currency  from prod_mod_pkg_pii.anc_offer_pnr_priority
# select *,
# regexp_replace(pass_id,'^0' ,'') as pass_id_clean,
#         CASE WHEN seg_flight_1 is null THEN 1
#             WHEN seg_flight_2 is null THEN 2
#             ELSE 3
#         END as numConnections
# from prod_mod_pkg_pii.anc_offer_pnr_priority 
# where Sale_Indicator = 1
# and Sale_Price != 0
# and PNR_CREATE_DT between '{start_date}' AND '{end_date}'

if test_run:
    pnr_sale_priority = spark.sql(f"""
        SELECT *,
            regexp_replace(pass_id, '^0', '') AS pass_id_clean,
            CASE WHEN seg_flight_1 IS NULL THEN 1
                 WHEN seg_flight_2 IS NULL THEN 2
                 ELSE 3
            END AS numConnections
        FROM prod_mod_pkg_pii.anc_offer_pnr_priority
        WHERE Sale_Indicator = 1
          AND Sale_Price != 0
          AND PNR_CREATE_DT BETWEEN '{start_date}' AND '{end_date}'
    """)
    pnr_sale_priority.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("rm_workspace.pnr_sale_priority")
    print("\u2713 pnr_sale_priority materialized")

spark.table("rm_workspace.pnr_sale_priority").createOrReplaceTempView("pnr_sale_priority")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
from datetime import date

if test_run:
    # Read from materialized tables
    temp_view_df = spark.table("temp_view")
    pnr_sale_df  = spark.table("pnr_sale_priority")

    # =========================
    # PRE-DEDUPLICATE RIGHT SIDE
    # =========================
    # pnr_sale_df has multiple rows per join key (different sale_DT / Sale_Price).
    # A direct left join explodes temp_view rows by the fan-out count before the
    # Sales flag and ROW_NUMBER can deduplicate — causing executor OOM on saveAsTable.
    # Collapsing to one row per (join key + Sale_Price) with the earliest sale_DT
    # eliminates the explosion while preserving all semantically distinct price matches.
    pnr_sale_dedup = pnr_sale_df.groupBy(
        "PNR", "pass_id_clean", "od_dep_dt", "od_origin", "od_destination",
        "os_displayPrice_currency", "numConnections", "Sale_Price"
    ).agg(F.min("sale_DT").alias("sale_DT"))

    # =========================
    # JOIN (sort-merge, deduped right side)
    # =========================
    joined = temp_view_df.alias("t").join(
        pnr_sale_dedup.alias("p"),
        on=[
            F.col("t.PNR") == F.col("p.PNR"),
            F.col("t.pass_id_clean") == F.col("p.pass_id_clean"),
            F.col("t.od_dep_dt") == F.col("p.od_dep_dt"),
            F.col("t.od_origin") == F.col("p.od_origin"),
            F.col("t.od_destination") == F.col("p.od_destination"),
            F.col("t.os_displayPrice_currency") == F.col("p.os_displayPrice_currency"),
            F.col("t.numConnections") == F.col("p.numConnections")
        ],
        how="left"
    )

    # =========================
    # SALES FLAG
    # =========================
    joined = joined.withColumn(
        "Sales",
        F.when(
            (F.col("p.sale_DT").cast("date") >= F.col("t.offer_date_utc").cast("date")) &
            (F.abs(F.col("p.Sale_Price") - F.col("t.os_displayPrice")) < 1),
            F.lit("Sale")
        ).otherwise(F.lit("No"))
    ).withColumn("sale_DT", F.col("p.sale_DT"))

    # Keep only necessary columns early
    t_cols = [F.col(f"t.{c}") for c in temp_view_df.columns]
    joined = joined.select(*t_cols, "sale_DT", "Sales")

    # Persist before split: prevents Spark executing the BHJ twice
    # (once for sale_df, once for no_df). Join runs once, both filters
    # read from the cached result.
    joined = joined.persist(StorageLevel.MEMORY_AND_DISK)

    # =========================
    # SPLIT BEFORE WINDOW
    # =========================
    sale_df = joined.filter(F.col("Sales") == "Sale")
    no_df   = joined.filter(F.col("Sales") == "No")

    # =========================
    # WINDOW ONLY ON SALES
    # =========================
    w = Window.partitionBy(
        "PNR",
        "pass_id_clean",
        "od_dep_dt",
        "od_origin",
        "od_destination"
    ).orderBy("offer_date_utc")

    sale_df = (
        sale_df
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # =========================
    # UNION BACK
    # =========================
    temp_link2 = sale_df.unionByName(no_df)

    # =========================
    # WRITE OUTPUT
    # =========================
    temp_link2.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("rm_workspace.transactionOfferPriority")

    joined.unpersist()

    temp_link2 = spark.table("rm_workspace.transactionOfferPriority")
    temp_link2.createOrReplaceTempView("transactionOfferPriority")

else:
    temp_link2 = spark.table("rm_workspace.transactionOfferPriority")
    temp_link2.createOrReplaceTempView("transactionOfferPriority")

print("\u2713 transactionOfferPriority ready")

In [0]:
# Currency conversion is preserved but the expensive sort-merge join is eliminated.
# For USD rows (the vast majority), adj_displayPrice = os_displayPrice anyway.
# By filtering priority_adj to non-USD rows only, the right side becomes tiny
# and Spark broadcasts it — no shuffle, no sort-merge.
# Increase shuffle partitions for the large window shuffles in cells 13 and 17.
# Default 200 → ~5M rows/task on 17 months of data; 2000 → ~500K rows/task.
spark.conf.set("spark.sql.shuffle.partitions", "2000")

if test_run:
    spark.sql("""
        WITH base AS (
            SELECT /*+ BROADCAST(p) */
                t.pnr, t.pass_id, t.datePartition, t.OD_dep_dt, t.od_origin, t.od_destination,
                t.TierStatusHighestPnr, t.numConnections, t.sale_DT, t.Sales,
                t.key_id, t.unique_id, t.transactionId, t.os_displayPrice, t.os_displayPrice_currency,
                CASE
                    WHEN t.os_displayPrice_currency = 'USD' AND t.os_displayPrice IS NOT NULL
                    THEN t.os_displayPrice
                    ELSE p.adj_displayPrice
                END AS adj_displayPrice_USD,
                p.base_price,
                p.surcharge,
                p.security_line,
                p.dow_time
            FROM transactionOfferPriority t
            LEFT JOIN (
                SELECT transactionID, key_id, unique_id, datePartition, numConnections, adj_displayPrice, base_price, surcharge, security_line, dow_time
                FROM priority_adj
                WHERE os_displayPrice_currency != 'USD' OR os_displayPrice IS NULL
            ) p
                ON  t.transactionId  = p.transactionID
                AND t.key_id         = p.key_id
                AND t.unique_id      = p.unique_id
                AND t.datePartition  = p.datePartition
                AND t.numConnections = p.numConnections
        )
        SELECT pnr, pass_id, datePartition, OD_dep_dt, od_origin, od_destination,
               TierStatusHighestPnr, numConnections, sale_DT, Sales,
               key_id, unique_id, transactionId, os_displayPrice, os_displayPrice_currency, adj_displayPrice_USD,base_price,surcharge,security_line,dow_time
            , CASE
                WHEN adj_displayPrice_USD < 15 THEN '10-15'
                WHEN adj_displayPrice_USD < 20 THEN '15-20'
                WHEN adj_displayPrice_USD < 25 THEN '20-25'
                WHEN adj_displayPrice_USD < 30 THEN '25-30'
                WHEN adj_displayPrice_USD < 35 THEN '30-35'
                WHEN adj_displayPrice_USD < 40 THEN '35-40'
                WHEN adj_displayPrice_USD < 45 THEN '40-45'
                WHEN adj_displayPrice_USD < 50 THEN '45-50'
                ELSE '50+'
            END AS price_bucket,
            CASE
                WHEN adj_displayPrice_USD < 15 THEN 1
                WHEN adj_displayPrice_USD < 20 THEN 2
                WHEN adj_displayPrice_USD < 25 THEN 3
                WHEN adj_displayPrice_USD < 30 THEN 4
                WHEN adj_displayPrice_USD < 35 THEN 5
                WHEN adj_displayPrice_USD < 40 THEN 6
                WHEN adj_displayPrice_USD < 45 THEN 7
                WHEN adj_displayPrice_USD < 50 THEN 8
                ELSE 9
            END AS price_bucket_order
        FROM base
    """).write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("rm_workspace.transactionOfferBase")
    print("\u2713 transactionOfferBase materialized")

spark.table("rm_workspace.transactionOfferBase") \
     .createOrReplaceTempView("transactionOfferBase")

In [0]:
%sql
SELECT 
    transactionId,
    key_id,
    unique_id,
    datePartition,
    CAST(REGEXP_REPLACE(
    MAX(CASE WHEN p_os_annotations_name LIKE '%Base_t%' THEN p_os_annotations_values END),
    '[^0-9.]', '') AS DOUBLE) AS base_price,
    CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Surcharge_t%' THEN p_os_annotations_values END) AS DOUBLE) AS surcharge,
    CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Security%' THEN p_os_annotations_values END) AS DOUBLE) AS security_line,
    CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%DOW%' THEN p_os_annotations_values END) AS DOUBLE) AS dow_time,
    CAST(MAX(CASE WHEN p_os_annotations_name LIKE '%Functionality%' THEN p_os_annotations_values END) AS DOUBLE) AS functionality
FROM prod_mod_gold_pii.annotative_price
WHERE product_type <> 'seats'
    AND (p_os_annotations_name LIKE '%Priority%' OR p_os_annotations_name LIKE '%priority%')
    AND datePartition BETWEEN 20250101 AND 20260611
GROUP BY transactionId, datePartition, key_id, unique_id

In [0]:
%sql
WITH multi_price_ods AS (
    SELECT od_origin, od_destination, COUNT(DISTINCT base_price) AS distinct_price_count
    FROM transactionOfferBase
    WHERE base_price IS NOT NULL
    GROUP BY od_origin, od_destination
    HAVING COUNT(DISTINCT base_price) > 1
),
ranked AS (
    SELECT
        t.od_origin, t.od_destination, t.base_price, t.transactionId, t.datePartition,
        m.distinct_price_count,
        ROW_NUMBER() OVER (
            PARTITION BY t.od_origin, t.od_destination, t.base_price
            ORDER BY t.transactionId
        ) AS rn
    FROM transactionOfferBase t
    INNER JOIN multi_price_ods m
        ON t.od_origin = m.od_origin AND t.od_destination = m.od_destination
    WHERE t.base_price IS NOT NULL
)
SELECT od_origin, od_destination, base_price, transactionId, datePartition, distinct_price_count
FROM ranked
WHERE rn <= distinct_price_count
ORDER BY od_origin, od_destination, base_price, rn

In [0]:
%sql
WITH multi_security_ods AS (
    SELECT od_origin, od_destination, COUNT(DISTINCT security_line) AS distinct_security_count
    FROM transactionOfferBase
    WHERE security_line IS NOT NULL
    GROUP BY od_origin, od_destination
    HAVING COUNT(DISTINCT security_line) > 1
),
ranked AS (
    SELECT
        t.od_origin, t.od_destination, t.security_line, t.transactionId, t.datePartition,
        m.distinct_security_count,
        ROW_NUMBER() OVER (
            PARTITION BY t.od_origin, t.od_destination, t.security_line
            ORDER BY t.transactionId
        ) AS rn
    FROM transactionOfferBase t
    INNER JOIN multi_security_ods m
        ON t.od_origin = m.od_origin AND t.od_destination = m.od_destination
    WHERE t.security_line IS NOT NULL
)
SELECT od_origin, od_destination, security_line, transactionId, datePartition, distinct_security_count
FROM ranked
WHERE rn <= distinct_security_count
ORDER BY od_origin, od_destination, security_line, rn

In [0]:
%sql
-- Option A: one row per (pnr, pass_id, od, price, datePartition, transactionId)
-- Join is already done in transactionOfferBase — only the window runs here.
CREATE OR REPLACE TEMP VIEW finalRawTransactionOfferSale AS
SELECT * EXCEPT (rn)
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY pnr, pass_id, od_origin, od_destination,
                   os_displayPrice, datePartition, transactionId
      ORDER BY Sales DESC, sale_DT DESC NULLS LAST
    ) AS rn
  FROM transactionOfferBase
)
WHERE rn = 1

In [0]:
# Materialize the join + window (Option A) BEFORE enrichment.
# This breaks the lazy view chain so cell 13 only runs the lightweight market joins,
# instead of re-executing the expensive ROW_NUMBER window over the full dataset.
if test_run:
    spark.sql("SELECT * FROM finalRawTransactionOfferSale") \
         .write.mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("rm_workspace.finalRawTransactionOfferSale")
    print("\u2713 finalRawTransactionOfferSale materialized")

# Re-point temp view to the materialized table so cell 12 reads from Delta, not the lazy view
spark.table("rm_workspace.finalRawTransactionOfferSale") \
     .createOrReplaceTempView("finalRawTransactionOfferSale")

In [0]:
%sql
-- market_segment is pre-materialized (see cell after cell 4) — no CTE needed here.
CREATE OR REPLACE TEMP VIEW finalTransactionOfferSale AS
SELECT /*+ BROADCAST(ms) */ f.*,
    o.Mileage,
    o.Market,
    ms.leg_dep_tm,
    ms.pnr_od_count,
    ms.avg_business_prob,
    ms.avg_bleisure_prob,
    ms.avg_vfr_prob,
    ms.avg_vacation_prob,
    ms.avg_personal_prob,
    ms.region,
    ms.origin_country,
    ms.destination_country,
    ms.market_traveler_segment,

    -- FlightDuration (mileage-based)
    CASE
      WHEN o.Mileage IS NULL OR o.Mileage < 0 THEN NULL
      WHEN o.Mileage / 500.0 <  2  THEN 'Ultra_Short'
      WHEN o.Mileage / 500.0 <  4  THEN 'Short'
      WHEN o.Mileage / 500.0 <  7  THEN 'Medium'
      WHEN o.Mileage / 500.0 < 10  THEN 'True_Long'
      ELSE 'Ultra_Long'
    END AS FlightDuration,

    -- Region group (geographic segmentation)
    CASE
      WHEN ms.region IN ('US48', 'CANADA', 'ALASKA') THEN 'Domestic'
      WHEN ms.region = 'HAWAII' THEN 'Hawaii'
      WHEN ms.region IN ('IATA CARIBBEAN', 'MEXICO', 'CENTRAL AMERICA', 'US CARIBBEAN', 'SOUTH AMERICA') THEN 'MCLA'
      WHEN ms.region = 'ATLANTIC' THEN 'Transatlantic'
      WHEN ms.region IN ('PACIFIC', 'INDIA') THEN 'Transpacific'
      ELSE NULL
    END AS region_group

FROM finalRawTransactionOfferSale f
LEFT JOIN od_mile_market o
  ON f.od_origin = o.Origin
  AND f.od_destination = o.Destination
LEFT JOIN market_segment ms
  ON f.od_origin = ms.od_origin
  AND f.od_destination = ms.od_destination

In [0]:
from datetime import date

if test_run:
    spark.sql("SELECT * FROM finalTransactionOfferSale") \
         .write.mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("rm_workspace.finalTransactionOfferSale")

finalTransactionOfferSale = spark.table("rm_workspace.finalTransactionOfferSale")
finalTransactionOfferSale.createOrReplaceTempView("finalTransactionOfferSale")
print("\u2713 finalTransactionOfferSale ready")

In [0]:
%sql
-- Option B: one row per (pnr, pass_id, od, price) — Sale row wins
-- Join is already done in transactionOfferBase — only the window runs here.
CREATE OR REPLACE TEMP VIEW finalRawTransactionOfferSale_B AS
SELECT * EXCEPT (rn)
FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY pnr, pass_id, od_origin, od_destination, os_displayPrice
      ORDER BY Sales DESC, sale_DT ASC NULLS LAST
    ) AS rn
  FROM transactionOfferBase
)
WHERE rn = 1

In [0]:
# Mirrors the Option A pattern in cell 11: materialize the join + window (Option B)
# BEFORE cell 15's enrichment view reads from it, so cell 16 only runs lightweight joins.
if test_run:
    spark.sql("SELECT * FROM finalRawTransactionOfferSale_B") \
         .write.mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("rm_workspace.finalRawTransactionOfferSale_B")
    print("\u2713 finalRawTransactionOfferSale_B materialized")

# Re-point temp view to the materialized table
spark.table("rm_workspace.finalRawTransactionOfferSale_B") \
     .createOrReplaceTempView("finalRawTransactionOfferSale_B")

In [0]:
%sql
-- market_segment is pre-materialized (see cell after cell 4) — no CTE needed here.
CREATE OR REPLACE TEMP VIEW finalTransactionOfferSale_B AS
SELECT /*+ BROADCAST(ms) */ f.*,
    o.Mileage,
    o.Market,
    ms.pnr_od_count,
    ms.avg_business_prob,
    ms.avg_bleisure_prob,
    ms.avg_vfr_prob,
    ms.avg_vacation_prob,
    ms.avg_personal_prob,
    ms.region,
    ms.origin_country,
    ms.destination_country,
    ms.market_traveler_segment,

    CASE
      WHEN o.Mileage IS NULL OR o.Mileage < 0 THEN NULL
      WHEN o.Mileage / 500.0 <  2  THEN 'Ultra_Short'
      WHEN o.Mileage / 500.0 <  4  THEN 'Short'
      WHEN o.Mileage / 500.0 <  7  THEN 'Medium'
      WHEN o.Mileage / 500.0 < 10  THEN 'True_Long'
      ELSE 'Ultra_Long'
    END AS FlightDuration,

    -- Region group (geographic segmentation)
    CASE
      WHEN ms.region IN ('US48', 'CANADA', 'ALASKA') THEN 'Domestic'
      WHEN ms.region = 'HAWAII' THEN 'Hawaii'
      WHEN ms.region IN ('IATA CARIBBEAN', 'MEXICO', 'CENTRAL AMERICA', 'US CARIBBEAN', 'SOUTH AMERICA') THEN 'MCLA'
      WHEN ms.region = 'ATLANTIC' THEN 'Transatlantic'
      WHEN ms.region IN ('PACIFIC', 'INDIA') THEN 'Transpacific'
      ELSE NULL
    END AS region_group

FROM finalRawTransactionOfferSale_B f
LEFT JOIN od_mile_market o
  ON f.od_origin = o.Origin
  AND f.od_destination = o.Destination
LEFT JOIN market_segment ms
  ON f.od_origin = ms.od_origin
  AND f.od_destination = ms.od_destination

In [0]:
from datetime import date

if test_run:
    spark.sql("SELECT * FROM finalTransactionOfferSale_B") \
         .write.mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable("rm_workspace.finalTransactionOfferSale_B")

finalTransactionOfferSale_B = spark.table("rm_workspace.finalTransactionOfferSale_B")
finalTransactionOfferSale_B.createOrReplaceTempView("finalTransactionOfferSale_B")
print("\u2713 finalTransactionOfferSale_B (Option B) ready")

In [0]:
%sql
select * from rm_workspace.finalTransactionOfferSale_B
where Sales <> 'No'

In [0]:
%sql
select * From transactionOfferBase

In [0]:
%sql
WITH base_with_market AS (
    SELECT
        t.od_origin,
        ms.destination_country,
        LEFT(t.datePartition,4) as year, 
        CASE WHEN ms.destination_country = 'US' THEN 'Domestic' ELSE 'International' END AS market_type,
        t.base_price,
        t.security_line
    FROM transactionOfferBase t
    LEFT JOIN market_segment ms
        ON t.od_origin = ms.od_origin AND t.od_destination = ms.od_destination
),
base_price_dist AS (
    SELECT
        od_origin, market_type,
        base_price AS value,
        year,
        COUNT(*)   AS cnt,
        SUM(COUNT(*)) OVER (PARTITION BY od_origin, market_type) AS origin_market_total
    FROM base_with_market
    WHERE base_price IS NOT NULL
    GROUP BY od_origin, market_type, base_price, year
),
security_dist AS (
    SELECT
        od_origin, market_type,
        security_line AS value,
        year,
        COUNT(*)      AS cnt,
        SUM(COUNT(*)) OVER (PARTITION BY od_origin, market_type) AS origin_market_total
    FROM base_with_market
    WHERE security_line IS NOT NULL
    GROUP BY od_origin, market_type, security_line, year
)
SELECT od_origin, market_type, year, 'base_price'    AS metric, value, cnt AS count, ROUND(100.0 * cnt / origin_market_total, 1) AS pct
FROM base_price_dist
UNION ALL
SELECT od_origin, market_type, year, 'security_line' AS metric, value, cnt AS count, ROUND(100.0 * cnt / origin_market_total, 1) AS pct
FROM security_dist
ORDER BY od_origin, market_type, year, metric, value

In [0]:
%sql
WITH price_counts AS (
    SELECT
        od_origin,
        od_destination,
        SUBSTRING(CAST(datePartition AS STRING), 1, 4) AS year,
        base_price,
        COUNT(*) AS cnt
    FROM transactionOfferBase
    WHERE base_price IS NOT NULL
    GROUP BY
        od_origin,
        od_destination,
        SUBSTRING(CAST(datePartition AS STRING), 1, 4),
        base_price
),

valid_od_years AS (
    SELECT
        od_origin,
        od_destination,
        year
    FROM price_counts
    GROUP BY
        od_origin,
        od_destination,
        year
    HAVING COUNT(DISTINCT base_price) > 1
)

SELECT
    p.od_origin,
    p.od_destination,
    p.year,
    p.base_price,
    p.cnt,
    ROUND(
        100.0 * p.cnt /
        SUM(p.cnt) OVER (
            PARTITION BY p.od_origin, p.od_destination, p.year
        ),
        2
    ) AS pct_count
FROM price_counts p
INNER JOIN valid_od_years v
    ON p.od_origin = v.od_origin
   AND p.od_destination = v.od_destination
   AND p.year = v.year
ORDER BY
    p.od_origin,
    p.od_destination,
    p.year,
    p.base_price;